# LiDAR Spatial Eval: compact tutorial

This notebook demonstrates the preprint's distance metrics and hard-points evaluation on a tiny synthetic cloud. No dataset download is required. Install the dependencies first with `python -m pip install -r requirements.txt` and run the notebook from the repository root.

Preprint (arXiv): [Spatially-Aware Evaluation Framework for Aerial LiDAR Point Cloud Semantic Segmentation](https://arxiv.org/abs/2603.22420)

In [ ]:
import numpy as np
import pandas as pd

from metrics import get_distance_metrics
from metrics.display import create_distance_table

We define one ground strip (`1`) and one building strip (`2`), followed by predictions from two models. Coordinates are expressed in meters.

In [ ]:
xyz = np.array([
    [0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [2.0, 0.0, 0.0],
    [0.0, 5.0, 0.0], [1.0, 5.0, 0.0], [2.0, 5.0, 0.0],
    [10.0, 0.0, 0.0], [10.0, 5.0, 0.0],
])
ground_truth = np.array([1, 1, 1, 2, 2, 2, 1, 2])

predictions = {
    "model_a": np.array([1, 1, 2, 2, 2, 2, 1, 1]),
    "model_b": np.array([1, 2, 1, 2, 1, 2, 1, 2]),
}

class_names = {1: "Ground", 2: "Building"}
thresholds = {1: 2.0, 2: 2.0}

The hard-points subset is the union of the indices misclassified by at least one model. Both models will be evaluated on these same points.

In [ ]:
hard_mask = np.any(
    np.stack([prediction != ground_truth for prediction in predictions.values()]),
    axis=0,
)
hard_indices = np.flatnonzero(hard_mask)
hard_indices

In [ ]:
tables = []
for model_name, prediction in predictions.items():
    metrics = get_distance_metrics(
        gt_classes=ground_truth,
        pred_classes=prediction,
        coords=xyz,
        classification_dict=class_names,
        class_distance_limits=thresholds,
        tp_distance=True,
        error_idx=hard_indices,
    )
    metrics["class_names"] = class_names
    table = create_distance_table(metrics)
    table.insert(0, "Model", model_name)
    tables.append(table)

pd.concat(tables, ignore_index=True)

`MDE` averages class-specific clipped distances; correct predictions inside the hard subset contribute zero. `rho` is the number of distant false positives divided by the number of hard points whose ground-truth class is the corresponding class. `mu` averages only non-distant misclassified points. Nearest ground-truth references are searched in the full cloud, not only in the hard subset.